# Figure 3: MOLLI, ShMOLLI and SASHA sampling

The cell below is tagged `#| label: fig-sampling-output`, so `04-molli-shmolli-sasha.md` embeds its output with `:::{figure} #fig-sampling-output`. The simulation is in `cardiac_sim.py` in this folder.

In [ ]:
#| label: fig-sampling-output
# Figure 3: how MOLLI, ShMOLLI and SASHA sample the recovery curve during the heartbeats.
# The simulation itself is in cardiac_sim.py (shared with Figures 4 and 5).
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import os, sys
sys.path.append(os.path.join(os.getcwd(), "notebooks"))  # find cardiac_sim.py also when run from the book folder
from cardiac_sim import SEQUENCES, simulate, fit

# Interactive figure for the website + PNG snapshot for the PDF (same probe as the template).
try:
    pio.to_image(go.Figure(), format="png")
    pio.renderers.default = "plotly_mimetype+png"
except Exception as exc:
    pio.renderers.default = "plotly_mimetype"
    print(f"Static PNG export unavailable ({type(exc).__name__}); PDF will show the caption only.")

RR = 1000.0            # time between two heartbeats [ms] (60 beats per minute)
TISSUES = {"myocardium (T1 = 1000 ms)": 1000.0, "blood (T1 = 1600 ms)": 1600.0,
           "myocardium after contrast (T1 = 400 ms)": 400.0}

fig = make_subplots(rows=2, cols=1, vertical_spacing=0.17,
                    subplot_titles=("What happens during the scan",
                                    "All images sorted by their time after the preparation pulse"))
options, TRACES = [], 6
for seq in SEQUENCES:
    for tissue, T1 in TISSUES.items():
        options.append((seq, tissue))
        vis = (seq == "MOLLI" and T1 == 1000.0)
        t_c, mz_c, events, samples = simulate(seq, T1, RR)
        t0 = t_c[0]
        # ECG: one R-peak marker per heartbeat
        ecg_x, ecg_y = [], []
        for b in np.arange(np.ceil(t0 / RR) * RR, t_c[-1], RR):
            ecg_x += [b - 40, b, b + 40, None]; ecg_y += [-1.35, -1.12, -1.35, None]
        fig.add_trace(go.Scatter(x=[(v - t0) / 1000 if v is not None else None for v in ecg_x],
                                 y=ecg_y, mode="lines", line=dict(color="gray"),
                                 name="heartbeat (ECG)", visible=vis), row=1, col=1)
        fig.add_trace(go.Scatter(x=(t_c - t0) / 1000, y=mz_c, mode="lines",
                                 line=dict(color="lightgray", width=3), name="Mz",
                                 visible=vis), row=1, col=1)
        px, py = [], []
        for t_ev, kind in events:
            if kind != "img":
                px += [(t_ev - t0) / 1000] * 2 + [None]; py += [-1.1, 1.1, None]
        is_inv = seq != "SASHA"
        fig.add_trace(go.Scatter(x=px, y=py, mode="lines",
                                 line=dict(color="royalblue" if is_inv else "darkorange", dash="dot"),
                                 name="inversion pulse" if is_inv else "saturation pulse",
                                 visible=vis), row=1, col=1)
        img_t = [(t_ev - t0) / 1000 for t_ev, kind in events if kind == "img"]
        fig.add_trace(go.Scatter(x=img_t, y=[s[1] for s in samples], mode="markers",
                                 marker=dict(size=9, color="black"), name="image",
                                 visible=vis), row=1, col=1)
        # bottom: images sorted by their effective time, plus the fit
        model, p, T1_fit, n_used = fit(seq, samples, RR)
        x = np.array([s[0] for s in samples])
        x_plot = np.where(np.isinf(x), 4400, x)       # SASHA's first image drawn at the right edge
        colors = ["black"] * n_used + ["darkorange"] * (len(x) - n_used)
        fig.add_trace(go.Scatter(x=x_plot, y=[s[1] for s in samples], mode="markers",
                                 marker=dict(size=9, color=colors),
                                 name="images (orange: not used in the fit)" if n_used < len(x) else "images",
                                 visible=vis), row=2, col=1)
        if seq == "SASHA":
            label = f"fit: T1 = {T1_fit:.0f} ms"
        else:
            label = f"fit: T1* = {p[2]:.0f} ms, corrected T1 = {T1_fit:.0f} ms"
        xf = np.linspace(0, 4500, 300)
        fig.add_trace(go.Scatter(x=xf, y=model(xf, *p), mode="lines",
                                 line=dict(color="crimson", dash="dash"),
                                 name=label + f" (true T1 = {T1:.0f} ms)", visible=vis), row=2, col=1)

buttons = []
for i, (seq, tissue) in enumerate(options):
    visible = [False] * (TRACES * len(options))
    visible[i * TRACES:(i + 1) * TRACES] = [True] * TRACES
    buttons.append(dict(label=f"{seq}, {tissue}", method="update", args=[{"visible": visible}]))

fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction="down", x=0, xanchor="left", y=1.18, yanchor="top")],
    height=750, legend=dict(orientation="h", y=-0.15), margin=dict(t=110),
    template="plotly_white",
)
fig.update_xaxes(title_text="Time [s]", row=1, col=1)
fig.update_yaxes(title_text="Mz / M0", range=[-1.4, 1.15], row=1, col=1)
fig.update_xaxes(title_text="Time after inversion (TI) or saturation (TS) [ms]", range=[0, 4600], row=2, col=1)
fig.update_yaxes(title_text="Mz / M0", range=[-1.1, 1.1], row=2, col=1)
fig
